# Quantizer smoke tests

This notebook runs small reconstruction tests for the local Leech lattice vector quantizer, an E8 lattice quantizer, the QTIP bitshift/trellis quantizer from `Cornell-RelaxML/qtip`, and a compact RaBitQ-style randomized binary quantizer.

The goal is a quick harness for checking APIs, rates, scales, reconstruction error, and runtime on synthetic Gaussian vectors. It is not a full LLM quantization evaluation.

In [1]:
from __future__ import annotations

import contextlib
import importlib.util
import io
import math
import subprocess
import sys
import time
import types
from dataclasses import dataclass
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import torch
    TORCH_AVAILABLE = True
    TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except Exception as exc:
    torch = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc
    TORCH_DEVICE = None

try:
    import pandas as pd
except Exception:
    pd = None

print(f"root: {ROOT}")
print(f"torch available: {TORCH_AVAILABLE}")
print(f"torch device: {TORCH_DEVICE}")


root: /home/diego/Documents/master/llm-quantization-benchmarks
torch available: True
torch device: cuda


In [2]:
# Benchmark configuration. Keep these small for CPU smoke tests.
SEED = 0
DIM = 24
N_SAMPLES = 1024
TARGET_BITS_PER_DIM = 2.0

# Side-information accounting. Set SCALE_BITS = 0 if the scale is fixed by convention.
# With one shared scale, the overhead is SCALE_BITS / (N_SAMPLES * DIM).
SCALE_BITS = 16
SCALE_GRANULARITY = "global"  # "global" or "per_vector"
PER_VECTOR_SCALAR_BITS = 16

# Shell 13 has 48 cumulative shape bits, i.e. exactly 2 bits/dim in 24-D.
# This is intended for the GPU Leech path; lower this for CPU-only smoke tests.
LEECH_MAX_SHELL = 13

# Cubic E8 radius 1.5 is the closest finite cubic-E8 setting under 2 bits/dim.
E8_RADIUS = 1.5

# QTIP source checkout. The notebook can clone this repo if it is missing.
QTIP_REPO = ROOT / "third_party" / "qtip"
AUTO_CLONE_QTIP = True
QTIP_GIT_URL = "https://github.com/Cornell-RelaxML/qtip.git"

# Small QTIP bitshift configuration. K is the nominal bits/dim target.
QTIP_CONFIG = dict(L=8, K=int(TARGET_BITS_PER_DIM), V=1, tlut_bits=8, decode_mode="quantlut")

rng = np.random.default_rng(SEED)
X = rng.standard_normal((N_SAMPLES, DIM)).astype(np.float32)
X.shape, X.dtype


((1024, 24), dtype('float32'))

In [3]:
@dataclass
class QuantizerReport:
    method: str
    rate_bits_per_dim: float
    mse: float
    rmse: float
    error_std: float
    squared_error_std: float
    sqnr_bits: float
    scale: float
    seconds: float
    quantize_seconds: float
    dequantize_seconds: float
    metadata: dict


def mse_to_sqnr_bits(mse: float) -> float:
    return float("inf") if mse == 0 else -0.5 * math.log2(mse)


def reconstruction_error_stats(x: np.ndarray, recon: np.ndarray) -> dict[str, float]:
    error = x - recon
    squared_error = error ** 2
    mse = float(np.mean(squared_error))
    return {
        "mse": mse,
        "rmse": float(math.sqrt(mse)),
        "error_std": float(np.std(error)),
        "squared_error_std": float(np.std(squared_error)),
    }


def synchronize_for_timing():
    if TORCH_AVAILABLE and torch.cuda.is_available():
        torch.cuda.synchronize()


def scale_overhead_bpd(x: np.ndarray, scale_bits: int, granularity: str) -> float:
    if scale_bits <= 0:
        return 0.0
    if granularity == "global":
        return float(scale_bits) / float(x.size)
    if granularity == "per_vector":
        return float(scale_bits) / float(x.shape[-1])
    raise ValueError(f"unknown scale granularity: {granularity}")


def optimize_global_scale(x: np.ndarray, quantize_normalized, scales: np.ndarray):
    best_scale = float(scales[0])
    best_mse = float("inf")
    best_recon = None
    for scale in scales:
        recon, mse = quantize_at_scale(x, quantize_normalized, float(scale))
        if mse < best_mse:
            best_scale = float(scale)
            best_mse = mse
            best_recon = recon
    return best_scale, best_mse, best_recon


def quantize_at_scale(x: np.ndarray, quantize_normalized, scale: float):
    recon = float(scale) * quantize_normalized(x / float(scale))
    stats = reconstruction_error_stats(x, recon)
    return recon, stats["mse"]


def choose_shared_scale(x: np.ndarray, quantizers, scales: np.ndarray):
    best_scale = float(scales[0])
    best_mean_mse = float("inf")
    for scale in scales:
        mses = [quantize_at_scale(x, item["quantize"], float(scale))[1] for item in quantizers]
        mean_mse = float(np.mean(mses))
        if mean_mse < best_mean_mse:
            best_scale = float(scale)
            best_mean_mse = mean_mse
    return best_scale, best_mean_mse


def run_scaled_quantizer(method: str, x: np.ndarray, rate: float, quantize_normalized, scales, metadata=None, shared_scale=None):
    metadata = {} if metadata is None else dict(metadata)
    start = time.perf_counter()
    if shared_scale is None:
        scale, mse, recon = optimize_global_scale(x, quantize_normalized, np.asarray(scales, dtype=np.float32))
        metadata.setdefault("scale_mode", "per_method_optimized")
    else:
        scale = float(shared_scale)
        recon, mse = quantize_at_scale(x, quantize_normalized, scale)
        metadata.setdefault("scale_mode", "shared")
    seconds = time.perf_counter() - start
    stats = reconstruction_error_stats(x, recon)
    report = QuantizerReport(
        method=method,
        rate_bits_per_dim=float(rate),
        mse=stats["mse"],
        rmse=stats["rmse"],
        error_std=stats["error_std"],
        squared_error_std=stats["squared_error_std"],
        sqnr_bits=mse_to_sqnr_bits(stats["mse"]),
        scale=float(scale),
        seconds=float(seconds),
        quantize_seconds=float(seconds),
        dequantize_seconds=0.0,
        metadata=metadata,
    )
    return report, recon


def run_split_scaled_quantizer(method: str, x: np.ndarray, rate: float, encode_normalized, decode_normalized, metadata=None, shared_scale=1.0):
    metadata = {} if metadata is None else dict(metadata)
    metadata.setdefault("scale_mode", "shared")
    scale = float(shared_scale)
    normalized = x / scale

    synchronize_for_timing()
    quantize_start = time.perf_counter()
    codes = encode_normalized(normalized)
    synchronize_for_timing()
    quantize_seconds = time.perf_counter() - quantize_start

    synchronize_for_timing()
    dequantize_start = time.perf_counter()
    recon_normalized = decode_normalized(codes)
    synchronize_for_timing()
    dequantize_seconds = time.perf_counter() - dequantize_start

    recon = scale * recon_normalized
    stats = reconstruction_error_stats(x, recon)
    report = QuantizerReport(
        method=method,
        rate_bits_per_dim=float(rate),
        mse=stats["mse"],
        rmse=stats["rmse"],
        error_std=stats["error_std"],
        squared_error_std=stats["squared_error_std"],
        sqnr_bits=mse_to_sqnr_bits(stats["mse"]),
        scale=scale,
        seconds=float(quantize_seconds + dequantize_seconds),
        quantize_seconds=float(quantize_seconds),
        dequantize_seconds=float(dequantize_seconds),
        metadata=metadata,
    )
    return report, recon


def reports_table(reports):
    rows = [
        {
            "method": r.method,
            "rate_bpd": r.rate_bits_per_dim,
            "side_info_bpd": float(r.metadata.get("side_info_bpd", 0.0)),
            "effective_rate_bpd": r.rate_bits_per_dim + float(r.metadata.get("side_info_bpd", 0.0)),
            "target_delta_bpd": r.rate_bits_per_dim + float(r.metadata.get("side_info_bpd", 0.0)) - TARGET_BITS_PER_DIM,
            "mse": r.mse,
            "rmse": r.rmse,
            "error_std": r.error_std,
            "squared_error_std": r.squared_error_std,
            "sqnr_bits": r.sqnr_bits,
            "scale": r.scale,
            "seconds": r.seconds,
            "quantize_seconds": r.quantize_seconds,
            "dequantize_seconds": r.dequantize_seconds,
            "metadata": r.metadata,
        }
        for r in reports
    ]
    if pd is not None:
        return pd.DataFrame(rows).sort_values(["mse", "seconds"]).reset_index(drop=True)
    return rows


## E8 lattice quantizer

This is a NumPy-only cubic E8 finite-codebook quantizer. It mirrors the local `CubicE8Quantizer` idea without requiring `torch` to import the helper module.

In [4]:
DIM_E8 = 8


def enumerate_cubic_e8(radius: float) -> np.ndarray:
    integer_values = range(-int(math.floor(radius)), int(math.floor(radius)) + 1)
    points = []

    def rec_integer(prefix, parity_sum):
        if len(prefix) == DIM_E8:
            if parity_sum % 2 == 0:
                points.append(tuple(float(v) for v in prefix))
            return
        for value in integer_values:
            rec_integer(prefix + [value], parity_sum + value)

    rec_integer([], 0)

    half_min = math.ceil(-radius - 0.5)
    half_max = math.floor(radius - 0.5)
    half_shifts = range(half_min, half_max + 1)

    def rec_half(prefix, parity_sum):
        if len(prefix) == DIM_E8:
            if parity_sum % 2 == 0:
                points.append(tuple(prefix))
            return
        for shift in half_shifts:
            value = shift + 0.5
            if abs(value) <= radius + 1e-12:
                rec_half(prefix + [value], parity_sum + shift)

    rec_half([], 0)
    return np.asarray(points, dtype=np.float32)


def quantize_blocks_to_codebook(blocks: np.ndarray, codebook: np.ndarray, chunk: int = 4096) -> np.ndarray:
    return codebook[encode_blocks_to_codebook(blocks, codebook, chunk=chunk)]


def encode_blocks_to_codebook(blocks: np.ndarray, codebook: np.ndarray, chunk: int = 4096) -> np.ndarray:
    out = np.empty((blocks.shape[0],), dtype=np.int64)
    codebook = codebook.astype(np.float32, copy=False)
    code_norm = np.sum(codebook * codebook, axis=1)
    for start in range(0, blocks.shape[0], chunk):
        stop = min(start + chunk, blocks.shape[0])
        b = blocks[start:stop].astype(np.float32, copy=False)
        distances = np.sum(b * b, axis=1, keepdims=True) + code_norm[None, :] - 2.0 * b @ codebook.T
        out[start:stop] = np.argmin(distances, axis=1)
    return out


class CubicE8NotebookQuantizer:
    def __init__(self, radius: float, device=None, chunk: int = 4096):
        self.radius = float(radius)
        self.codebook = enumerate_cubic_e8(self.radius)
        if self.codebook.size == 0:
            raise ValueError("empty E8 codebook")
        self.rate = math.log2(self.codebook.shape[0]) / DIM_E8
        self.chunk = int(chunk)
        self.device = device if TORCH_AVAILABLE and device is not None else None
        self.backend = "gpu" if self.device is not None and self.device.type == "cuda" else "cpu"
        self.codebook_t = None
        self.code_norm_t = None
        if self.device is not None:
            self.codebook_t = torch.as_tensor(self.codebook, dtype=torch.float32, device=self.device)
            self.code_norm_t = torch.sum(self.codebook_t * self.codebook_t, dim=1)

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        if samples.shape[-1] % DIM_E8 != 0:
            raise ValueError("last dimension must be divisible by 8")
        blocks = samples.reshape(-1, DIM_E8)
        output_shape = samples.shape[:-1] + (samples.shape[-1] // DIM_E8,)
        if self.codebook_t is None:
            return encode_blocks_to_codebook(blocks, self.codebook, chunk=self.chunk).reshape(output_shape)

        x = torch.as_tensor(blocks, dtype=torch.float32, device=self.device)
        out = torch.empty((x.shape[0],), dtype=torch.long, device=self.device)
        with torch.no_grad():
            for start in range(0, x.shape[0], self.chunk):
                stop = min(start + self.chunk, x.shape[0])
                b = x[start:stop]
                distances = torch.sum(b * b, dim=1, keepdim=True) + self.code_norm_t[None, :] - 2.0 * b @ self.codebook_t.T
                out[start:stop] = torch.argmin(distances, dim=1)
        return out.reshape(output_shape)

    def decode_normalized(self, indices) -> np.ndarray:
        if torch is not None and torch.is_tensor(indices):
            with torch.no_grad():
                recon = self.codebook_t[indices.to(self.device).long()].reshape(*indices.shape[:-1], indices.shape[-1] * DIM_E8)
            return recon.detach().cpu().numpy().astype(np.float32)
        return self.codebook[np.asarray(indices, dtype=np.int64)].reshape(*indices.shape[:-1], indices.shape[-1] * DIM_E8)


## Leech lattice vector quantizer

This wraps the local `leech_lattice_vector_quantizer` module. If CUDA is available, it uses the batched GPU implementation; otherwise it falls back to the CPU implementation. The current CPU implementation prints the winning codeword during quantization, so the wrapper suppresses that output for table runs.

In [5]:
class LeechNotebookQuantizer:
    def __init__(self, max_shell: int):
        self.max_shell = max_shell
        self.backend = "cpu"
        self.device = "cpu"

        if TORCH_AVAILABLE and torch.cuda.is_available():
            try:
                from leech_lattice_vector_quantizer.leech_lattice_vector_quantizer_gpu import LeechLatticeVectorQuantizerGpu

                self.quantizer = LeechLatticeVectorQuantizerGpu(max_shell=max_shell, verbose=False, device="cuda")
                # Trigger NVRTC compilation here so setup errors are reported before the benchmark loop.
                _ = self.quantizer.quantize(torch.zeros((1, 24), dtype=torch.float32, device=self.quantizer.device))
                self.backend = "gpu"
                self.device = str(self.quantizer.device)
            except Exception as exc:
                print(f"Leech GPU unavailable, falling back to CPU: {type(exc).__name__}: {exc}")
                from leech_lattice_vector_quantizer.leech_lattice_vector_quantizer import LeechLatticeVectorQuantizer

                self.quantizer = LeechLatticeVectorQuantizer(max_shell=max_shell, verbose=False)
        else:
            from leech_lattice_vector_quantizer.leech_lattice_vector_quantizer import LeechLatticeVectorQuantizer

            self.quantizer = LeechLatticeVectorQuantizer(max_shell=max_shell, verbose=False)
        self.rate = self.quantizer.shape_bits / 24.0

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        if samples.shape[-1] != 24:
            raise ValueError("Leech quantizer expects 24-D rows")
        if self.backend == "gpu":
            x = torch.as_tensor(samples, dtype=torch.float32, device=self.quantizer.device)
            with torch.no_grad():
                return self.quantizer.quantize(x)

        indices = np.empty(samples.shape[:-1], dtype=np.int64)
        for i, row in enumerate(samples):
            with contextlib.redirect_stdout(io.StringIO()):
                indices[i] = self.quantizer.quantize(row)
        return indices

    def decode_normalized(self, indices) -> np.ndarray:
        if self.backend == "gpu":
            with torch.no_grad():
                recon = self.quantizer.dequantize(indices)
            return recon.detach().cpu().numpy().astype(np.float32)

        out = np.empty((*np.asarray(indices).shape, 24), dtype=np.float32)
        for i, idx in enumerate(np.asarray(indices).reshape(-1)):
            out.reshape(-1, 24)[i] = np.asarray(self.quantizer.dequantize(int(idx)), dtype=np.float32)
        return out


## QTIP bitshift quantizer

The QTIP repository imports compiled kernel bindings at package import time. For this notebook's codebook-only test, the loader below installs minimal stubs and imports `lib/codebook/bitshift.py` directly. Install `torch` first; the notebook can clone `Cornell-RelaxML/qtip` into `third_party/qtip` if missing.

In [6]:
def ensure_qtip_repo(repo_path: Path, auto_clone: bool = True) -> Path | None:
    if repo_path.exists():
        return repo_path
    if not auto_clone:
        return None
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", QTIP_GIT_URL, str(repo_path)], check=True)
    return repo_path


def load_qtip_bitshift_module(qtip_repo: Path):
    if not TORCH_AVAILABLE:
        raise RuntimeError(f"torch is required for QTIP: {TORCH_IMPORT_ERROR}")

    sys.modules.setdefault("lib", types.ModuleType("lib"))

    codebook_stub = types.ModuleType("lib.codebook")
    codebook_stub.kdict = {}
    sys.modules["lib.codebook"] = codebook_stub

    sys.modules.setdefault("lib.utils", types.ModuleType("lib.utils"))

    kernel_check = types.ModuleType("lib.utils.kernel_check")
    kernel_check.has_kernel = lambda *args, **kwargs: False
    sys.modules["lib.utils.kernel_check"] = kernel_check

    kernel_decompress = types.ModuleType("lib.utils.kernel_decompress")

    def _decode_compressed_not_used(*args, **kwargs):
        raise RuntimeError("kernel decompression is not used in this codebook-only notebook test")

    kernel_decompress.decode_compressed = _decode_compressed_not_used
    sys.modules["lib.utils.kernel_decompress"] = kernel_decompress

    matmul_had = types.ModuleType("lib.utils.matmul_had")
    matmul_had.matmul_hadU_cuda = lambda *args, **kwargs: None
    matmul_had.matmul_hadUt_cuda = lambda *args, **kwargs: None
    sys.modules["lib.utils.matmul_had"] = matmul_had

    module_path = qtip_repo / "lib" / "codebook" / "bitshift.py"
    spec = importlib.util.spec_from_file_location("qtip_bitshift_notebook", module_path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(module_path)

    # The codebook smoke test runs fine in eager mode. QTIP decorates one helper with
    # torch.compile, which requires a C++ compiler through Inductor even on CPU.
    # Temporarily disabling the decorator keeps this notebook portable.
    original_compile = torch.compile
    torch.compile = lambda fn=None, *args, **kwargs: (lambda f: f) if fn is None else fn
    try:
        spec.loader.exec_module(module)
    finally:
        torch.compile = original_compile
    return module


class QTIPNotebookQuantizer:
    def __init__(self, repo_path: Path, config: dict, device=None):
        module = load_qtip_bitshift_module(repo_path)
        self.config = dict(config)
        self.device = device if device is not None else torch.device("cpu")
        self.backend = "gpu" if self.device.type == "cuda" else "cpu"
        self.cb = module.bitshift_codebook(**self.config).to(self.device)
        self.cb.fakeinf = self.cb.fakeinf.to(self.device)
        self.rate = float(self.config["K"])

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        x = torch.as_tensor(samples, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            x_t = x.T.contiguous().to(torch.float16)
            T = x_t.shape[0]
            roll_x = torch.roll(x_t, T // (2 * self.cb.V) * self.cb.V, 0)
            states = self.cb.quantize_seq(roll_x, overlap=None)
            overlap = states[T // (2 * self.cb.V)] >> self.cb.K * self.cb.V
            states = self.cb.quantize_seq(x_t, overlap=overlap)
            states = states.T.contiguous().to(self.device)
        self.last_states = states
        return states

    def decode_normalized(self, states) -> np.ndarray:
        states = states.to(self.device) if torch.is_tensor(states) else torch.as_tensor(states, device=self.device)
        with torch.no_grad():
            states_t = states.T.contiguous()
            batch = states.shape[0]
            dim = states.shape[1] * self.cb.V
            recon = self.cb.recons(states_t).transpose(0, 1).reshape(dim, batch).T.contiguous()
        self.last_states = states
        return recon.detach().cpu().numpy().astype(np.float32)


## Uniform scalar quantizer

A clipped per-coordinate uniform scalar quantizer is included as a simple exact-rate baseline.

In [7]:
class UniformScalarNotebookQuantizer:
    def __init__(self, bits: int, device=None):
        if bits < 1:
            raise ValueError("bits must be positive")
        self.bits = int(bits)
        self.levels = 1 << self.bits
        self.qmin = -(self.levels // 2)
        self.qmax = self.levels // 2 - 1
        self.rate = float(self.bits)
        self.device = device if TORCH_AVAILABLE and device is not None else None
        self.backend = "gpu" if self.device is not None and self.device.type == "cuda" else "cpu"

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        if self.device is None:
            return np.clip(np.round(samples), self.qmin, self.qmax).astype(np.int16)
        x = torch.as_tensor(samples, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            return torch.clamp(torch.round(x), self.qmin, self.qmax).to(torch.int16)

    def decode_normalized(self, codes) -> np.ndarray:
        if torch is not None and torch.is_tensor(codes):
            return codes.to(torch.float32).detach().cpu().numpy()
        return np.asarray(codes, dtype=np.float32)


## RaBitQ-style randomized binary quantizer

This is a self-contained reconstruction test inspired by RaBitQ: apply a fixed random orthogonal rotation, encode one sign bit per dimension on the unit sphere, use a per-vector least-squares scale, and rotate back. It is useful as a 1-bit smoke-test baseline, not a replacement for the official ANN distance-estimation implementation.

In [8]:
def random_orthogonal(dim: int, rng: np.random.Generator) -> np.ndarray:
    h = rng.standard_normal((dim, dim)).astype(np.float32)
    q, r = np.linalg.qr(h)
    signs = np.sign(np.diag(r))
    signs[signs == 0] = 1
    return (q * signs).astype(np.float32)


class RaBitQStyleQuantizer:
    def __init__(self, dim: int, seed: int = 0, device=None):
        self.dim = int(dim)
        self.rotation = random_orthogonal(self.dim, np.random.default_rng(seed))
        self.rate = 1.0
        self.device = device if TORCH_AVAILABLE and device is not None else None
        self.backend = "gpu" if self.device is not None and self.device.type == "cuda" else "cpu"
        self.rotation_t = None
        if self.device is not None:
            self.rotation_t = torch.as_tensor(self.rotation, dtype=torch.float32, device=self.device)

    def quantize_normalized(self, samples: np.ndarray) -> np.ndarray:
        return self.decode_normalized(self.encode_normalized(samples))

    def encode_normalized(self, samples: np.ndarray):
        if samples.shape[-1] != self.dim:
            raise ValueError(f"expected {self.dim}-D rows")
        if self.rotation_t is None:
            z = samples @ self.rotation
            sign_bits = z >= 0
            signs = np.where(sign_bits, 1.0, -1.0).astype(np.float32) / math.sqrt(self.dim)
            alpha = np.sum(z * signs, axis=1, keepdims=True)
            return sign_bits, alpha.astype(np.float32)

        x = torch.as_tensor(samples, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            z = x @ self.rotation_t
            sign_bits = z >= 0
            signs = torch.where(sign_bits, 1.0, -1.0).to(torch.float32) / math.sqrt(self.dim)
            alpha = torch.sum(z * signs, dim=1, keepdim=True)
        return sign_bits, alpha.to(torch.float32)

    def decode_normalized(self, codes) -> np.ndarray:
        sign_bits, alpha = codes
        if torch is not None and torch.is_tensor(sign_bits):
            sign_bits = sign_bits.to(self.device)
            alpha = alpha.to(self.device)
            with torch.no_grad():
                signs = torch.where(sign_bits, 1.0, -1.0).to(torch.float32) / math.sqrt(self.dim)
                recon_rot = alpha * signs
                recon = recon_rot @ self.rotation_t.T
            return recon.detach().cpu().numpy().astype(np.float32)

        signs = np.where(sign_bits, 1.0, -1.0).astype(np.float32) / math.sqrt(self.dim)
        recon_rot = alpha * signs
        return (recon_rot @ self.rotation.T).astype(np.float32)


## Run tests

In [9]:
reports = []
reconstructions = {}
scale_grid = np.linspace(0.05, 3.0, 80, dtype=np.float32)
quantizer_items = []
shared_scale_side_info_bpd = scale_overhead_bpd(X, SCALE_BITS, SCALE_GRANULARITY)
quantizer_device = TORCH_DEVICE if TORCH_AVAILABLE else None

uniform = UniformScalarNotebookQuantizer(bits=int(TARGET_BITS_PER_DIM), device=quantizer_device)
quantizer_items.append({
    "method": f"Uniform scalar 2-bit ({uniform.backend})",
    "rate": uniform.rate,
    "quantize": uniform.quantize_normalized,
    "encode": uniform.encode_normalized,
    "decode": uniform.decode_normalized,
    "metadata": {
        "target_bpd": TARGET_BITS_PER_DIM,
        "bits": uniform.bits,
        "levels": uniform.levels,
        "backend": uniform.backend,
        "device": str(uniform.device),
    },
})

e8 = CubicE8NotebookQuantizer(E8_RADIUS, device=quantizer_device)
quantizer_items.append({
    "method": f"E8 cubic ({e8.backend})",
    "rate": e8.rate,
    "quantize": e8.quantize_normalized,
    "encode": e8.encode_normalized,
    "decode": e8.decode_normalized,
    "metadata": {
        "target_bpd": TARGET_BITS_PER_DIM,
        "radius": E8_RADIUS,
        "codebook_size": int(e8.codebook.shape[0]),
        "backend": e8.backend,
        "device": str(e8.device),
    },
})

leech = LeechNotebookQuantizer(LEECH_MAX_SHELL)
quantizer_items.append({
    "method": f"Leech shells <= {LEECH_MAX_SHELL} ({leech.backend})",
    "rate": leech.rate,
    "quantize": leech.quantize_normalized,
    "encode": leech.encode_normalized,
    "decode": leech.decode_normalized,
    "metadata": {
        "target_bpd": TARGET_BITS_PER_DIM,
        "backend": leech.backend,
        "device": leech.device,
        "total_count": int(leech.quantizer.total_count),
        "shape_bits": int(leech.quantizer.shape_bits),
    },
})

rabitq = RaBitQStyleQuantizer(DIM, seed=SEED, device=quantizer_device)
quantizer_items.append({
    "method": f"RaBitQ-style binary 1-bit ({rabitq.backend})",
    "rate": rabitq.rate,
    "quantize": rabitq.quantize_normalized,
    "encode": rabitq.encode_normalized,
    "decode": rabitq.decode_normalized,
    "metadata": {
        "target_bpd": TARGET_BITS_PER_DIM,
        "rotation_seed": SEED,
        "per_vector_scalar_bits": PER_VECTOR_SCALAR_BITS,
        "per_vector_scalar_bpd": PER_VECTOR_SCALAR_BITS / DIM,
        "backend": rabitq.backend,
        "device": str(rabitq.device),
    },
})

qtip_status = "not run"
if TORCH_AVAILABLE:
    try:
        qtip_repo = ensure_qtip_repo(QTIP_REPO, auto_clone=AUTO_CLONE_QTIP)
        qtip = QTIPNotebookQuantizer(qtip_repo, QTIP_CONFIG, device=quantizer_device)
        quantizer_items.append({
            "method": f"QTIP bitshift ({qtip.backend})",
            "rate": qtip.rate,
            "quantize": qtip.quantize_normalized,
            "encode": qtip.encode_normalized,
            "decode": qtip.decode_normalized,
            "metadata": {
                "target_bpd": TARGET_BITS_PER_DIM,
                "backend": qtip.backend,
                "device": str(qtip.device),
                **QTIP_CONFIG,
            },
        })
        qtip_status = f"run ({qtip.backend}, {qtip.device})"
    except Exception as exc:
        qtip_status = f"skipped: {type(exc).__name__}: {exc}"
else:
    qtip_status = f"skipped: torch unavailable: {TORCH_IMPORT_ERROR}"

shared_scale, shared_scale_mean_mse = choose_shared_scale(X, quantizer_items, scale_grid)
print(f"shared scale: {shared_scale:.6g} (mean calibration mse: {shared_scale_mean_mse:.6g})")
print(f"shared scale side info: {shared_scale_side_info_bpd:.6g} bpd ({SCALE_BITS} bits, {SCALE_GRANULARITY})")
print(f"QTIP status: {qtip_status}")

for item in quantizer_items:
        side_info_bpd = shared_scale_side_info_bpd + float(item["metadata"].get("per_vector_scalar_bpd", 0.0))
        report, recon = run_split_scaled_quantizer(
            item["method"],
            X,
            item["rate"],
            item["encode"],
            item["decode"],
            metadata={
                "side_info_bpd": side_info_bpd,
                "shared_scale_bits": SCALE_BITS,
                "shared_scale_granularity": SCALE_GRANULARITY,
                "shared_scale_mean_mse": shared_scale_mean_mse,
                **item["metadata"],
            },
            shared_scale=shared_scale,
        )
        reports.append(report)
        reconstructions[report.method] = recon

table = reports_table(reports)
table


shared scale: 1.02089 (mean calibration mse: 0.171621)
shared scale side info: 0.000651042 bpd (16 bits, global)
QTIP status: run (gpu, cuda)


,method,rate_bpd,side_info_bpd,effective_rate_bpd,target_delta_bpd,mse,rmse,error_std,squared_error_std,sqnr_bits,scale,seconds,quantize_seconds,dequantize_seconds,metadata
0,Leech shells <= 13 (gpu),2.000000,0.000651,2.000651,0.000651,0.077367,0.278149,0.278130,0.110013,1.846071,1.020886,0.674846,0.674280,0.000566,"{'side_info_bpd': 0.0006510416666666666, 'shar..."
1,E8 cubic (gpu),1.892209,0.000651,1.892860,-0.107140,0.120262,0.346788,0.346788,0.253994,1.527874,1.020886,0.004312,0.004001,0.000311,"{'side_info_bpd': 0.0006510416666666666, 'shar..."
2,Uniform scalar 2-bit (gpu),2.000000,0.000651,2.000651,0.000651,0.149310,0.386406,0.381922,0.345303,1.371811,1.020886,0.000195,0.000123,0.000072,"{'side_info_bpd': 0.0006510416666666666, 'shar..."
3,QTIP bitshift (gpu),2.000000,0.000651,2.000651,0.000651,0.164104,0.405097,0.400934,0.320503,1.303660,1.020886,0.008680,0.008545,0.000135,"{'side_info_bpd': 0.0006510416666666666, 'shar..."
4,RaBitQ-style binary 1-bit (gpu),1.000000,0.667318,1.667318,-0.332682,0.347064,0.589121,0.589119,0.503322,0.763364,1.020886,0.000438,0.000283,0.000155,"{'side_info_bpd': 0.6673177083333333, 'shared_..."


In [10]:
# Inspect the first vector reconstruction from each quantizer.
np.set_printoptions(precision=4, suppress=True)
print("original[0]:")
print(X[0])
for name, recon in reconstructions.items():
    print(f"\n{name} reconstruction[0]:")
    print(recon[0])
    print(f"row mse: {np.mean((X[0] - recon[0]) ** 2):.6f}")


original[0]:
[ 0.1257 -0.1321  0.6404  0.1049 -0.5357  0.3616  1.304   0.9471 -0.7037
 -1.2654 -0.6233  0.0413 -2.325  -0.2188 -1.2459 -0.7323 -0.5443 -0.3163
  0.4116  1.0425 -0.1285  1.3665 -0.6652  0.3515]

Uniform scalar 2-bit (gpu) reconstruction[0]:
[ 0.      0.      1.0209  0.     -1.0209  0.      1.0209  1.0209 -1.0209
 -1.0209 -1.0209  0.     -2.0418  0.     -1.0209 -1.0209 -1.0209  0.
  0.      1.0209  0.      1.0209 -1.0209  0.    ]
row mse: 0.087757

E8 cubic (gpu) reconstruction[0]:
[ 0.      0.      1.0209  0.     -1.0209  0.      1.0209  1.0209 -0.5104
 -1.5313 -0.5104  0.5104 -1.5313 -0.5104 -1.5313 -0.5104 -0.5104 -0.5104
  0.5104  1.5313 -0.5104  1.5313 -0.5104  0.5104]
row mse: 0.097381

Leech shells <= 13 (gpu) reconstruction[0]:
[ 0.3609  0.3609  1.0828  0.3609 -0.3609  0.3609  1.0828  1.0828 -0.3609
 -1.0828 -1.0828  0.3609 -2.5266 -0.3609 -1.0828 -1.0828 -0.3609 -0.3609
  0.3609  1.0828 -0.3609  1.0828 -0.3609  0.3609]
row mse: 0.066624

RaBitQ-style binary 1-bit